In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Library

In [2]:
# 데이터 전처리
import pandas as pd
import numpy as np
import gzip
import json
import copy
import os

# 작업 진행률 시각화
from tqdm import tqdm
tqdm.pandas()

# 이미지 처리
from skimage import io
import requests
from PIL import Image
from io import BytesIO

# tensorflow 모델
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing import image
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import img_to_array, load_img
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.optimizers import Adam

# Utils

In [3]:
# Helper: URL에서 이미지 로드 & (224×224)으로 리사이즈
def load_and_preprocess(url, idx=None, target_size=(224,224)):
    try:
        response = requests.get(url, stream=True, timeout=180)
        img = Image.open(BytesIO(response.content)).resize(target_size)
        img = img.convert("RGB")
        img_array = img_to_array(img)
        img_array = preprocess_input(img_array)
        return img_array
    except Exception as e:
        print(f"[{idx}] Error fetching image: {e}")
        # 에러 인덱스 기록
        if idx is not None:
            error_indices.append(idx)
        img_array = np.zeros((*target_size, 3), dtype=np.float32)
        img_array = preprocess_input(img_array)
        return img_array

In [4]:
# batch 단위로 feature 추출
def predict_batch(url_list, index_list, batch_size=4, model=None):

    # 1. 이미지 불러오기 및 전처리
    image_arrays = [load_and_preprocess(url, idx) for url, idx in zip(url_list, index_list)]
    batch_array = np.stack(image_arrays, axis=0)

    # 2. 배치 단위로 예측
    preds = model.predict(batch_array, batch_size=batch_size, verbose=1)
    return preds

In [5]:
# Top 5 core Filtering
def top_n_filtering(data, chk=5):
    cnt = 1
    while True:
        # 사용자 필터링
        user_counts = data['user_id'].value_counts()
        users_to_remove = user_counts[user_counts < chk].index
        data = data[~data['user_id'].isin(users_to_remove)]

        # 아이템 필터링
        item_counts = data['asin'].value_counts()
        items_to_remove = item_counts[item_counts < chk].index
        data = data[~data['asin'].isin(items_to_remove)]

        print(f"Epoch {cnt}")
        print(f" - Remaining users: {data['user_id'].nunique()}")
        print(f" - Remaining items: {data['asin'].nunique()}")
        print(f" - Remaining reviews: {len(data)}")

        # 두 조건 모두 만족 시 종료
        if (len(users_to_remove) == 0) and (len(items_to_remove) == 0):
            break

        cnt += 1

    return data

# Load Dataset

In [6]:
df = pd.read_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Cell_Phones_lgm_rec_v1.pkl')
print(df.shape)
df.head()

(23491, 7)


,user_id,asin,text,rating,title_y,medium_image_url,sentence_transformer_emb
0,AHUPTBY3F3UN2S5H7K5JLP6MAV5Q,B0BJKYJR3L,"Nice privacy shield. Good color, fits well, et...",3,Jephuwneh 2 Pack Gradient Tempered Glass for i...,https://m.media-amazon.com/images/I/515oPLx13k...,"[-0.08926809, 0.12239321, 0.25131106, -0.01834..."
1,AHUPTBY3F3UN2S5H7K5JLP6MAV5Q,B0BHTZQHQW,"This thing is great. It's not too big, has a d...",5,"NEOCOR Magnetic Wireless Power Bank, 10,000mAh...",https://m.media-amazon.com/images/I/71K8BDxzsA...,"[-0.18675853, 0.3855072, -0.038075477, 0.37468..."
2,AHUPTBY3F3UN2S5H7K5JLP6MAV5Q,B0B9N4VRB9,This is a much better way to mount my iPhone (...,5,Lamicall Gooseneck Phone Holder Bed - [Newest ...,https://m.media-amazon.com/images/I/71b+Sx-Jq7...,"[0.07915155, -0.087157056, 0.4343377, -0.44102..."
3,AHUPTBY3F3UN2S5H7K5JLP6MAV5Q,B0B93H178F,"This is a fantastic charger, especially if you...",5,"Shargeek USB C Charger 140W Wall Charger, Mult...",https://m.media-amazon.com/images/I/61R6UXrAKT...,"[0.07165233, 0.28588203, -0.05228097, 0.242655..."
4,AFERCDY2EFJKT7QUQ75GISNHTFOQ,B0BPXZWCBG,There never seems to be enough charging cords ...,4,"Charging Station for Multiple Devices,60W 12A ...",https://m.media-amazon.com/images/I/51fqzioGKe...,"[-0.11567296, 0.17700651, -0.100320175, 0.1198..."


# Extract VGG16 Image Embedding

In [6]:
# Load ResNet50 model
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3), pooling='max')

x = Flatten()(base_model.output)
predictions = Dense(4096, activation='relu')(x)
model = Model(inputs=base_model.input, outputs=predictions)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


In [8]:
# Apply the function to each row
batch_size = 8
all_image_urls = df['medium_image_url'].tolist()
all_indices = df.index.tolist()  # 실제 DataFrame index

error_indices = []
results = []
for i in tqdm(range(0, len(all_image_urls), batch_size)):
    batch_urls = all_image_urls[i:i+batch_size]
    batch_indices = all_indices[i:i+batch_size]
    preds = predict_batch(batch_urls, batch_indices, batch_size=len(batch_urls), model=model)
    results.extend(preds)

embedding = pd.DataFrame({'ResNet50_emb': [e for e in results]})
df = pd.concat([df, embedding], axis=1)

  0%|          | 0/2937 [00:00<?, ?it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step


  0%|          | 1/2937 [00:07<6:05:45,  7.47s/it]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step


  0%|          | 2/2937 [00:09<3:24:11,  4.17s/it]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step


... (5,865 outputs left out)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step


100%|█████████▉| 2936/2937 [1:25:07<00:01,  1.23s/it]

1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step


100%|██████████| 2937/2937 [1:25:12<00:00,  1.74s/it]


In [9]:
# error_indices에 오류난 데이터 인덱스가 다 들어있음
print(f"\n불러오지 못한 이미지 인덱스: {error_indices}")
print(f"\n총 오류 개수: {len(error_indices)}")
print("\n실패한 url:")

for idx in error_indices:
    print(df.loc[idx, 'medium_image_url'])


불러오지 못한 이미지 인덱스: [1706, 2214]

총 오류 개수: 2

실패한 url:
https://m.media-amazon.com/images/I/61q3mhgiSuL._SL800_.jpg
https://m.media-amazon.com/images/I/61HS8FKPClL._SL800_.jpg


In [10]:
len(error_indices)

2

In [16]:
# 행 인덱스와 컬럼명을 이용해 직접 할당
# df.at[1706, 'ResNet50_emb'] = results[0]
# df.at[2214, 'ResNet50_emb'] = results[1]

In [17]:
df.to_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Cell_Phones_lgm_rec_v2.pkl')
df.to_json('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Cell_Phones_lgm_rec_v2.json')